# Task 3.1: Two-Component Ablation
**Student:** Bhavishya Sahay | Roll No: 230071

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.mixture import GaussianMixture
from scipy.optimize import linear_sum_assignment
from scipy.special import logsumexp
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
K = 3; P_NEIGHBORS = 10; LAMBDA = 0.1; MAX_ITER = 100; TOL = 1e-4; REG = 1e-4

data = load_wine()
X_raw, y_true = data.data, data.target
from sklearn.preprocessing import StandardScaler
X = StandardScaler().fit_transform(X_raw)

def clustering_accuracy(y_true, y_pred):
    n = max(y_true.max(), y_pred.max()) + 1
    cost = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cost[t][p] += 1
    row, col = linear_sum_assignment(-cost)
    return cost[row, col].sum() / len(y_true)

def build_knn_graph(X, p=P_NEIGHBORS):
    from sklearn.neighbors import kneighbors_graph
    G = kneighbors_graph(X, n_neighbors=p, mode='connectivity', include_self=False).toarray()
    return np.maximum(G, G.T)

def log_gaussian(x, mu, Sigma):
    d = len(mu)
    diff = x - mu
    try:
        sign, logdet = np.linalg.slogdet(Sigma)
        if sign <= 0: return -1e10
        return -0.5*(d*np.log(2*np.pi) + logdet + diff @ np.linalg.inv(Sigma) @ diff)
    except: return -1e10

def run_lcgmm(X, K=K, p=P_NEIGHBORS, lam=LAMBDA, use_knn=True, use_kl=True, max_iter=MAX_ITER, tol=TOL, reg=REG):
    N, D = X.shape
    W = build_knn_graph(X, p) if use_knn else np.zeros((N,N))
    rows, cols = np.where(W > 0)
    gmm_init = GaussianMixture(n_components=K, random_state=SEED, reg_covar=reg)
    gmm_init.fit(X)
    pi = gmm_init.weights_.copy(); mu = gmm_init.means_.copy()
    Sigma = gmm_init.covariances_.copy() + reg*np.eye(D)
    prev_obj = -np.inf
    for it in range(max_iter):
        log_resp = np.zeros((N,K))
        for k in range(K):
            for i in range(N):
                log_resp[i,k] = np.log(pi[k]+1e-300) + log_gaussian(X[i], mu[k], Sigma[k])
        log_norm = logsumexp(log_resp, axis=1, keepdims=True)
        R = np.exp(log_resp - log_norm)
        Nk = R.sum(0)+1e-10; pi = Nk/N
        for k in range(K):
            rk = R[:,k]
            mu_std = (rk[:,None]*X).sum(0)/Nk[k]
            reg_mu = np.zeros(D)
            for i,j in zip(rows,cols):
                if use_kl: reg_mu += (rk[i]-rk[j])*(X[i]-X[j])
                else:
                    w_ij = np.linalg.norm(R[i]-R[j])
                    reg_mu += w_ij*(X[i]-X[j])*np.sign(rk[i]-rk[j]+1e-10)
            mu[k] = mu_std - lam*reg_mu/(2*Nk[k])
            diff = X-mu[k]
            S_std = (rk[:,None,None]*diff[:,:,None]*diff[:,None,:]).sum(0)/Nk[k]
            reg_S = np.zeros((D,D))
            for i,j in zip(rows,cols):
                Si = np.outer(X[i]-mu[k],X[i]-mu[k]); Sj = np.outer(X[j]-mu[k],X[j]-mu[k])
                if use_kl: reg_S += (rk[i]-rk[j])*(Si-Sj)
                else:
                    w_ij = np.linalg.norm(R[i]-R[j])
                    reg_S += w_ij*(Si-Sj)*np.sign(rk[i]-rk[j]+1e-10)
            Sigma[k] = S_std - lam*reg_S/(2*Nk[k]) + reg*np.eye(D)
        obj = log_norm.sum()
        if abs(obj-prev_obj)<tol: break
        prev_obj=obj
    return R.argmax(1)

print("Setup complete.")

Setup complete.


## Ablation A: Removing the kNN Graph (Local Consistency Regulariser)

The kNN graph W (Equation 4) is the structural backbone of LCGMM. Without it, the regularisation term R (Equation 5) becomes zero because there are no edges to sum over, and the M-step updates for μₖ and Σₖ (Equations 13–14) reduce exactly to the standard GMM updates. The graph is what allows the method to detect and exploit manifold structure — removing it converts LCGMM back into a vanilla GMM.

In [2]:
# ============================================================
# ABLATION A: Remove the kNN graph (Component 1)
# ============================================================
# When use_knn=False, rows and cols are empty, so reg_mu=0 and reg_S=0.
# The M-step reduces exactly to standard GMM — Eq.(13) and (14) with lambda*correction=0.

full_labels = run_lcgmm(X, use_knn=True, use_kl=True)
full_acc = clustering_accuracy(y_true, full_labels)

abl_a_labels = run_lcgmm(X, use_knn=False, use_kl=True)
abl_a_acc = clustering_accuracy(y_true, abl_a_labels)

print(f"Full LCGMM (with kNN graph): {full_acc*100:.1f}%")
print(f"Ablation A  (no kNN graph) : {abl_a_acc*100:.1f}%")
print(f"Difference: {full_acc*100 - abl_a_acc*100:+.1f}%")

Full LCGMM (with kNN graph): 93.8%
Ablation A  (no kNN graph) : 98.3%
Difference: -4.5%


The code above runs LCGMM twice: once with the full kNN graph and once with no graph (rows and cols are empty, so reg_mu and reg_S remain zero). Both use identical initialisation and EM procedure — the only difference is whether the regularisation term is active.

In [3]:
fig, ax = plt.subplots(figsize=(6, 5))
methods = ['Full LCGMM\n(kNN + KL reg)', 'Ablation A\n(no kNN graph)']
accs = [full_acc*100, abl_a_acc*100]
bars = ax.bar(methods, accs, color=['#2196F3','#FF5722'], edgecolor='black', linewidth=0.8, width=0.4)
ax.set_ylim(0, 110)
ax.set_ylabel('Clustering Accuracy (%)', fontsize=12)
ax.set_title('Ablation A: Effect of Removing kNN Graph\n(Local Consistency Regulariser)', fontsize=11, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{acc:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax.annotate(f'Diff: {full_acc*100-abl_a_acc*100:+.1f}%', xy=(0.5,0.87), xycoords='axes fraction', ha='center', fontsize=13, color='red', fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/bhavishyamac/Desktop/230071-midesm/partB/results/task3_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to partB/results/task3_ablation_A.png")

Saved to partB/results/task3_ablation_A.png


**Interpretation:** Removing the kNN graph actually *increases* accuracy by 4.5% on the Wine dataset (98.3% vs 93.8%). This is a meaningful and somewhat surprising result that reveals an important property: **the Wine dataset does not have the kind of manifold structure that LCGMM is designed to exploit**. Wine's 13 chemical features produce three clusters that are well-separated in Euclidean space (as confirmed by GMM's 96.6% accuracy). When LCGMM adds the kNN regulariser, it subtly distorts the cluster boundaries to enforce smoothness along the graph — but since the graph neighbours are not all from the same class, this introduces noise rather than geometric structure. This reveals that the regulariser's contribution is highly dataset-dependent: its value is positive only when the data has genuine manifold structure (like face images or digit images), and can be harmful when clusters are already Euclidean-Gaussian. The paper's own results support this: LCGMM also loses to GMM on the Waveform dataset (75.3% vs 76.3%).

**What this reveals:** The kNN graph is a double-edged contribution. It provides the critical improvement on manifold-structured data (MNIST, Chart, Vowel) but introduces a bias that hurts on well-separated Gaussian data. This suggests the method's success is highly dependent on the match between the dataset's true geometry and the manifold assumption.


## Ablation B: Replacing KL Divergence with L2 Distance

The paper specifically argues that using **KL-Divergence** (Equations 2–3) to measure dissimilarity between P(c|xᵢ) and P(c|xⱼ) is a key design choice — it is 'more natural' for comparing distributions and leads to a 'nice EM algorithm'. In this ablation, we replace the KL-derived weighting in the M-step update with the L2 norm of the difference between posterior vectors: ||R[i] − R[j]||₂, which is a simpler but less principled metric for distribution distance.

In [4]:
# ============================================================
# ABLATION B: Replace KL Divergence with L2 distance (Component 2)
# ============================================================
# The paper specifically chooses KL-Divergence (Eq. 2-3) to measure
# dissimilarity between distributions P(c|x_i) and P(c|x_j).
# Here we replace it with the L2 norm ||R[i] - R[j]||_2 as the weight.

abl_b_labels = run_lcgmm(X, use_knn=True, use_kl=False)
abl_b_acc = clustering_accuracy(y_true, abl_b_labels)

print(f"Full LCGMM   (KL divergence): {full_acc*100:.1f}%")
print(f"Ablation B   (L2 distance)  : {abl_b_acc*100:.1f}%")
print(f"Difference: {full_acc*100 - abl_b_acc*100:+.1f}%")

Full LCGMM   (KL divergence): 93.8%
Ablation B   (L2 distance)  : 63.5%
Difference: +30.3%


The code replaces KL-derived weights `(rk[i] - rk[j])` in the M-step with `||R[i]-R[j]||_2 * sign(rk[i]-rk[j])`. All other aspects of the algorithm (graph structure, EM loop, initialisation) are unchanged.

In [5]:
fig, ax = plt.subplots(figsize=(6,5))
methods = ['Full LCGMM\n(KL Divergence)', 'Ablation B\n(L2 Distance)']
accs = [full_acc*100, abl_b_acc*100]
bars = ax.bar(methods, accs, color=['#2196F3','#FF9800'], edgecolor='black', linewidth=0.8, width=0.4)
ax.set_ylim(0, 110)
ax.set_ylabel('Clustering Accuracy (%)', fontsize=12)
ax.set_title('Ablation B: KL Divergence vs L2 Distance\n(Regulariser Dissimilarity Metric)', fontsize=11, fontweight='bold')
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{acc:.1f}%', ha='center', fontsize=13, fontweight='bold')
ax.annotate(f'Diff: {full_acc*100-abl_b_acc*100:+.1f}%', xy=(0.5,0.87), xycoords='axes fraction', ha='center', fontsize=13, color='darkorange', fontweight='bold')
plt.tight_layout()
plt.savefig('/Users/bhavishyamac/Desktop/230071-midesm/partB/results/task3_ablation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to partB/results/task3_ablation_B.png")

Saved to partB/results/task3_ablation_B.png


**Interpretation:** Replacing KL divergence with L2 distance causes a dramatic drop of **30.3%** in clustering accuracy (63.5% vs 93.8%). This is by far the larger of the two ablation effects, and it reveals that **the choice of KL divergence as the distribution distance metric is critical to LCGMM's correctness**. The KL-derived weighting in Equations 13–14 comes from a principled derivation of the KL divergence gradient with respect to the Gaussian parameters (Section "M-step", Equation 10). This derivation yields clean closed-form updates that are consistent with the EM framework. The L2-based weighting, by contrast, does not correspond to any principled gradient, leading to incoherent parameter updates that destabilise the M-step. The magnitude of this ablation result (30+ points) is much larger than expected and highlights that the mathematical consistency between the KL divergence metric and the Gaussian parametric family is not just a matter of elegance — it is essential for the algorithm to converge to meaningful cluster solutions. The paper's claim that 'by using KL-Divergence, the new objective function can be solved more effectively by ordinary EM algorithm' is validated strongly by this ablation.
